[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_07/2_sql.ipynb)

# Day 7: SQL (mock exam)

**Mock exam rules.** Set one timer for **30 minutes** for the whole notebook. Work in order, but skip a question after 7 minutes and come back. Do not open any hint or solution until the timer ends. Then grade yourself: a question counts as solved only if the result matches exactly (columns, rows, order). Put every miss in the mistake log.

**How to work:** Read the tables first (columns, one row = what?). Write the query in your head in the order FROM, WHERE, GROUP BY, HAVING, SELECT, ORDER BY. Check the result for duplicates and NULLs before you open the solution.

**Today (30 min):**

| # | Question | Level | Minutes |
|---|---|---|---|
| Q1 | Newer employees | easy | 3 |
| Q2 | Most prolific composers | easy | 4 |
| Q3 | Revenue per support rep | easy | 5 |
| Q4 | Categories with dead videos | easy | 5 |
| Q5 | Active users by country | easy | 5 |
| Q6 | Customers who never bought Metal | easy | 8 |

**Handbook:** [SQL](https://mahsa-agz.github.io/ds-handbook/#sql), [cheat sheets](https://mahsa-agz.github.io/ds-handbook/#cheat)

## Tables

**Real tables: Chinook** (a digital music store, 2009 to 2013; MIT license, lerocha/chinook-database):

| Table | One row is | Key columns |
|---|---|---|
| `Customer` | one customer (59) | `CustomerId`, `FirstName`, `LastName`, `Company` (often NULL), `City`, `State` (often NULL), `Country`, `Email`, `SupportRepId` |
| `Employee` | one employee (8) | `EmployeeId`, `FirstName`, `LastName`, `Title`, `ReportsTo` (manager id, NULL for the boss), `HireDate`, `City` |
| `Invoice` | one purchase (412) | `InvoiceId`, `CustomerId`, `InvoiceDate`, `BillingCity`, `BillingCountry`, `Total` |
| `InvoiceLine` | one track on an invoice (2,240) | `InvoiceLineId`, `InvoiceId`, `TrackId`, `UnitPrice`, `Quantity` |
| `Track` | one song or video (3,503) | `TrackId`, `Name`, `AlbumId`, `MediaTypeId`, `GenreId`, `Composer` (can be NULL), `Milliseconds`, `Bytes`, `UnitPrice` |
| `Album` / `Artist` | one album (347) / artist (275) | `AlbumId`, `Title`, `ArtistId` / `ArtistId`, `Name` |
| `Genre` / `MediaType` | one genre (25) / format (5) | `GenreId`, `Name` / `MediaTypeId`, `Name` |
| `Playlist` / `PlaylistTrack` | one playlist (18) / one track in a playlist (8,715) | `PlaylistId`, `Name` / `PlaylistId`, `TrackId` |

**Made-up product tables** (a small short-video app, invented for practice):

| Table | One row is | Columns |
|---|---|---|
| `users` | one user | `user_id`, `signup_date` ('YYYY-MM-DD'), `country` (can be NULL), `platform` (ios / android / web) |
| `videos` | one uploaded video | `video_id`, `creator_id` (a user_id), `upload_date`, `category`, `duration_sec` |
| `events` | one user action on a video | `event_id`, `user_id`, `video_id`, `event_type` (view / like / comment / share), `event_time` ('YYYY-MM-DD HH:MM:SS'), `watch_sec` (views only, else NULL) |

The notebook uses SQLite (built into Python and Colab). Write standard SQL; the solutions note where PostgreSQL differs. Run a query with `run("""SELECT ...""")`.

## Setup

Run this cell once before the questions.

In [ ]:
import sqlite3
import pandas as pd

def run(q, con=None):
    """Runs a query and shows the result as a table (SELECT / WITH); other statements just run."""
    con = con or db
    if not q.strip():
        print("Write your query between the triple quotes, then run the cell.")
        return None
    if q.strip().lower().startswith(("select", "with")):
        return pd.read_sql_query(q, con)
    con.executescript(q)
    print("done")

# Finds the data folder (local copy of the repository, or /content/data in Colab) and downloads a missing file
# from the ds-handbook repository on GitHub (or from its original source).
import os, sys, urllib.request

RAW = "https://raw.githubusercontent.com/mahsa-agz/mahsa-agz.github.io/main/ds-handbook/exam_prep/data/"

def _find_data():
    if "google.colab" in sys.modules:
        os.makedirs("/content/data", exist_ok=True)
        return "/content/data"
    for p in ("../data", "data", "exam_prep/data"):
        if os.path.isdir(p):
            return p
    os.makedirs("data", exist_ok=True)
    return "data"

DATA = _find_data()
print("data folder:", DATA)

def data_path(name, url=None):
    """Local path of a data file; downloads it from the repository (or from url) if it is not there yet."""
    path = os.path.join(DATA, name)
    if not os.path.exists(path):
        for source in (RAW + name, url):
            if not source:
                continue
            try:
                print("downloading", name)
                urllib.request.urlretrieve(source, path)
                break
            except Exception as e:
                print("  could not download from", source, "->", e)
    return path

db = sqlite3.connect(':memory:')

# Real data: the Chinook digital music store (copied into the in-memory db).
_src = sqlite3.connect(data_path("chinook.sqlite", "https://github.com/lerocha/chinook-database/releases/download/v1.4.5/Chinook_Sqlite.sqlite"))
_src.backup(db)
_src.close()

# Made-up short-video product tables (users, videos, events): small, for exam-style questions.
db.executescript("""
CREATE TABLE users (
    user_id     INTEGER PRIMARY KEY,
    signup_date TEXT NOT NULL,      -- 'YYYY-MM-DD'
    country     TEXT,               -- NULL when unknown
    platform    TEXT NOT NULL       -- 'ios', 'android' or 'web'
);
CREATE TABLE videos (
    video_id     INTEGER PRIMARY KEY,
    creator_id   INTEGER NOT NULL REFERENCES users(user_id),
    upload_date  TEXT NOT NULL,     -- 'YYYY-MM-DD'
    category     TEXT NOT NULL,
    duration_sec INTEGER NOT NULL
);
CREATE TABLE events (
    event_id   INTEGER PRIMARY KEY,
    user_id    INTEGER NOT NULL REFERENCES users(user_id),
    video_id   INTEGER NOT NULL REFERENCES videos(video_id),
    event_type TEXT NOT NULL,       -- 'view', 'like', 'comment' or 'share'
    event_time TEXT NOT NULL,       -- 'YYYY-MM-DD HH:MM:SS'
    watch_sec  INTEGER              -- seconds watched; only for 'view', NULL otherwise
);
INSERT INTO users VALUES
(1, '2024-03-11', 'BR', 'web'),
(2, '2024-03-02', 'KR', 'ios'),
(3, '2024-03-12', 'US', 'android'),
(4, '2024-03-02', 'KR', 'web'),
(5, '2024-03-14', 'KR', 'android'),
(6, '2024-03-03', 'KR', 'ios'),
(7, '2024-03-19', NULL, 'android'),
(8, '2024-03-19', 'US', 'web'),
(9, '2024-03-02', 'ID', 'ios'),
(10, '2024-03-18', 'BR', 'android'),
(11, '2024-03-14', 'BR', 'ios'),
(12, '2024-03-19', 'DE', 'android'),
(13, '2024-03-04', 'ID', 'android'),
(14, '2024-03-04', 'KR', 'ios'),
(15, '2024-03-07', 'US', 'web'),
(16, '2024-03-11', NULL, 'web'),
(17, '2024-03-12', 'DE', 'android'),
(18, '2024-03-06', 'ID', 'ios'),
(19, '2024-03-19', 'DE', 'web'),
(20, '2024-03-11', 'US', 'android');
INSERT INTO videos VALUES
(101, 12, '2024-02-22', 'comedy', 90),
(102, 9, '2024-02-25', 'food', 30),
(103, 9, '2024-03-04', 'comedy', 120),
(104, 2, '2024-03-15', 'education', 90),
(105, 5, '2024-03-01', 'pets', 45),
(106, 12, '2024-03-06', 'education', 60),
(107, 2, '2024-02-22', 'food', 60),
(108, 14, '2024-03-12', 'comedy', 15),
(109, 14, '2024-03-13', 'food', 120),
(110, 12, '2024-03-12', 'gaming', 45),
(111, 14, '2024-03-03', 'pets', 45),
(112, 2, '2024-03-05', 'food', 30),
(113, 12, '2024-02-23', 'gaming', 15),
(114, 3, '2024-03-15', 'food', 30),
(115, 14, '2024-02-27', 'gaming', 60);
INSERT INTO events VALUES
(1, 9, 107, 'view', '2024-03-02 12:41:59', 4),
(2, 9, 107, 'like', '2024-03-02 12:42:16', NULL),
(3, 2, 107, 'view', '2024-03-02 14:06:14', 14),
(4, 2, 107, 'like', '2024-03-02 14:06:44', NULL),
(5, 9, 105, 'view', '2024-03-03 09:17:05', 28),
(6, 2, 101, 'view', '2024-03-03 09:53:42', 51),
(7, 2, 107, 'view', '2024-03-03 09:54:49', 52),
(8, 2, 107, 'comment', '2024-03-03 09:55:54', NULL),
(9, 6, 105, 'view', '2024-03-03 22:58:43', 4),
(10, 6, 105, 'share', '2024-03-03 22:58:52', NULL),
(11, 6, 101, 'view', '2024-03-04 11:00:30', 64),
(12, 6, 101, 'like', '2024-03-04 11:01:38', NULL),
(13, 6, 111, 'view', '2024-03-04 11:02:01', 35),
(14, 6, 111, 'like', '2024-03-04 11:02:51', NULL),
(15, 14, 105, 'view', '2024-03-04 15:34:40', 25),
(16, 14, 105, 'like', '2024-03-04 15:35:17', NULL),
(17, 13, 102, 'view', '2024-03-04 19:46:25', 2),
(18, 14, 101, 'view', '2024-03-05 12:39:47', 39),
(19, 6, 105, 'view', '2024-03-05 22:01:18', 6),
(20, 2, 112, 'view', '2024-03-06 07:09:37', 27),
(21, 6, 103, 'view', '2024-03-06 09:37:05', 97),
(22, 18, 103, 'view', '2024-03-06 21:13:43', 52),
(23, 18, 107, 'view', '2024-03-06 21:15:19', 42),
(24, 18, 107, 'like', '2024-03-06 21:16:10', NULL),
(25, 18, 107, 'share', '2024-03-06 21:16:13', NULL),
(26, 15, 101, 'view', '2024-03-07 11:44:01', 7),
(27, 15, 101, 'like', '2024-03-07 11:44:10', NULL),
(28, 15, 107, 'view', '2024-03-07 11:44:19', 14),
(29, 14, 106, 'view', '2024-03-08 08:14:09', 41),
(30, 13, 111, 'view', '2024-03-08 08:35:09', 7),
(31, 9, 112, 'view', '2024-03-08 13:23:34', 8),
(32, 9, 112, 'like', '2024-03-08 13:23:58', NULL),
(33, 9, 112, 'comment', '2024-03-08 13:24:12', NULL),
(34, 9, 111, 'view', '2024-03-08 13:24:57', 4),
(35, 6, 112, 'view', '2024-03-08 18:14:31', 14),
(36, 6, 112, 'like', '2024-03-08 18:14:46', NULL),
(37, 14, 101, 'view', '2024-03-09 08:01:03', 74),
(38, 14, 101, 'comment', '2024-03-09 08:02:29', NULL),
(39, 13, 102, 'view', '2024-03-09 12:33:10', 5),
(40, 13, 105, 'view', '2024-03-09 12:34:11', 21),
(41, 13, 105, 'like', '2024-03-09 12:34:34', NULL),
(42, 6, 107, 'view', '2024-03-09 20:22:24', 9),
(43, 6, 107, 'comment', '2024-03-09 20:22:44', NULL),
(44, 2, 103, 'view', '2024-03-10 10:33:47', 57),
(45, 16, 106, 'view', '2024-03-11 07:23:13', 18),
(46, 16, 105, 'view', '2024-03-11 07:24:16', 31),
(47, 16, 105, 'like', '2024-03-11 07:25:07', NULL),
(48, 1, 106, 'view', '2024-03-11 12:28:25', 58),
(49, 1, 106, 'like', '2024-03-11 12:29:37', NULL),
(50, 20, 101, 'view', '2024-03-11 12:34:38', 22),
(51, 20, 106, 'view', '2024-03-11 12:35:47', 55),
(52, 6, 102, 'view', '2024-03-11 16:16:23', 14),
(53, 6, 102, 'share', '2024-03-11 16:16:51', NULL),
(54, 18, 111, 'view', '2024-03-11 19:33:14', 31),
(55, 18, 111, 'like', '2024-03-11 19:33:50', NULL),
(56, 18, 111, 'share', '2024-03-11 19:33:57', NULL),
(57, 18, 105, 'view', '2024-03-11 19:34:32', 11),
(58, 20, 101, 'view', '2024-03-12 07:01:39', 89),
(59, 20, 111, 'view', '2024-03-12 07:03:19', 32),
(60, 20, 111, 'share', '2024-03-12 07:04:05', NULL),
(61, 16, 110, 'view', '2024-03-12 11:55:53', 44),
(62, 16, 106, 'view', '2024-03-12 11:57:26', 39),
(63, 16, 106, 'like', '2024-03-12 11:58:16', NULL),
(64, 14, 105, 'view', '2024-03-12 12:08:00', 11),
(65, 14, 105, 'comment', '2024-03-12 12:08:16', NULL),
(66, 14, 106, 'view', '2024-03-12 12:09:11', 27),
(67, 14, 106, 'share', '2024-03-12 12:09:56', NULL),
(68, 13, 112, 'view', '2024-03-12 14:39:25', 29),
(69, 13, 112, 'like', '2024-03-12 14:40:10', NULL),
(70, 6, 106, 'view', '2024-03-12 15:06:03', 42),
(71, 17, 105, 'view', '2024-03-12 16:27:17', 8),
(72, 17, 105, 'comment', '2024-03-12 16:27:32', NULL),
(73, 17, 101, 'view', '2024-03-12 16:28:06', 3),
(74, 3, 110, 'view', '2024-03-12 20:07:25', 22),
(75, 3, 110, 'like', '2024-03-12 20:07:55', NULL),
(76, 2, 103, 'view', '2024-03-13 15:34:26', 9),
(77, 2, 110, 'view', '2024-03-13 15:35:17', 28),
(78, 20, 110, 'view', '2024-03-13 18:21:30', 37),
(79, 6, 112, 'view', '2024-03-13 20:56:01', 14),
(80, 6, 102, 'view', '2024-03-13 20:57:06', 3),
(81, 13, 105, 'view', '2024-03-14 10:09:15', 4),
(82, 13, 107, 'view', '2024-03-14 10:10:06', 9),
(83, 1, 103, 'view', '2024-03-14 11:05:11', 31),
(84, 1, 103, 'comment', '2024-03-14 11:06:01', NULL),
(85, 15, 103, 'view', '2024-03-14 16:30:06', 14),
(86, 5, 103, 'view', '2024-03-14 17:46:31', 38),
(87, 5, 103, 'share', '2024-03-14 17:47:26', NULL),
(88, 5, 108, 'view', '2024-03-14 17:48:11', 13),
(89, 6, 107, 'view', '2024-03-15 11:10:30', 23),
(90, 6, 107, 'like', '2024-03-15 11:11:02', NULL),
(91, 3, 103, 'view', '2024-03-15 11:16:56', 61),
(92, 3, 103, 'like', '2024-03-15 11:18:01', NULL),
(93, 3, 104, 'view', '2024-03-15 11:18:48', 22),
(94, 2, 114, 'view', '2024-03-15 12:38:00', 27),
(95, 2, 114, 'like', '2024-03-15 12:38:32', NULL),
(96, 2, 101, 'view', '2024-03-15 12:39:12', 43),
(97, 20, 108, 'view', '2024-03-15 16:22:52', 8),
(98, 20, 108, 'like', '2024-03-15 16:23:09', NULL),
(99, 20, 111, 'view', '2024-03-15 16:23:27', 33),
(100, 1, 106, 'view', '2024-03-15 18:39:36', 10),
(101, 1, 112, 'view', '2024-03-15 18:40:34', 3),
(102, 17, 104, 'view', '2024-03-15 19:00:47', 57),
(103, 17, 104, 'view', '2024-03-15 19:02:43', 87),
(104, 20, 109, 'view', '2024-03-16 08:25:46', 115),
(105, 3, 101, 'view', '2024-03-16 09:46:23', 45),
(106, 3, 101, 'share', '2024-03-16 09:47:19', NULL),
(107, 3, 110, 'view', '2024-03-16 09:47:57', 20),
(108, 3, 110, 'comment', '2024-03-16 09:48:25', NULL),
(109, 2, 101, 'view', '2024-03-16 14:12:17', 14),
(110, 15, 114, 'view', '2024-03-16 16:03:45', 13),
(111, 15, 105, 'view', '2024-03-16 16:04:33', 41),
(112, 15, 105, 'share', '2024-03-16 16:05:31', NULL),
(113, 15, 104, 'view', '2024-03-17 08:34:36', 13),
(114, 15, 104, 'view', '2024-03-17 08:35:27', 38),
(115, 15, 104, 'share', '2024-03-17 08:36:21', NULL),
(116, 3, 103, 'view', '2024-03-17 08:57:49', 36),
(117, 3, 105, 'view', '2024-03-17 08:59:13', 27),
(118, 3, 105, 'like', '2024-03-17 08:59:57', NULL),
(119, 1, 111, 'view', '2024-03-17 19:06:30', 27),
(120, 1, 111, 'like', '2024-03-17 19:07:00', NULL),
(121, 6, 111, 'view', '2024-03-17 19:07:10', 12),
(122, 1, 106, 'view', '2024-03-17 19:07:12', 40),
(123, 6, 111, 'like', '2024-03-17 19:07:39', NULL),
(124, 1, 106, 'like', '2024-03-17 19:07:53', NULL),
(125, 6, 108, 'view', '2024-03-17 19:07:58', 7),
(126, 20, 111, 'view', '2024-03-17 22:38:49', 5),
(127, 14, 101, 'view', '2024-03-18 08:03:34', 53),
(128, 14, 101, 'like', '2024-03-18 08:04:33', NULL),
(129, 14, 101, 'comment', '2024-03-18 08:04:37', NULL),
(130, 14, 101, 'share', '2024-03-18 08:04:55', NULL),
(131, 10, 112, 'view', '2024-03-18 10:26:31', 16),
(132, 10, 112, 'like', '2024-03-18 10:26:52', NULL),
(133, 20, 111, 'view', '2024-03-18 13:02:42', 31),
(134, 6, 106, 'view', '2024-03-18 17:35:05', 17),
(135, 3, 105, 'view', '2024-03-18 20:57:04', 3),
(136, 3, 105, 'share', '2024-03-18 20:57:15', NULL),
(137, 20, 103, 'view', '2024-03-19 07:23:55', 102),
(138, 20, 103, 'like', '2024-03-19 07:25:46', NULL),
(139, 7, 105, 'view', '2024-03-19 08:41:45', 10),
(140, 18, 103, 'view', '2024-03-19 15:27:43', 63),
(141, 18, 103, 'like', '2024-03-19 15:28:55', NULL),
(142, 18, 108, 'view', '2024-03-19 15:29:20', 9),
(143, 12, 105, 'view', '2024-03-19 17:04:25', 17),
(144, 17, 112, 'view', '2024-03-19 17:16:40', 5),
(145, 17, 111, 'view', '2024-03-19 17:17:35', 12),
(146, 17, 111, 'like', '2024-03-19 17:18:01', NULL),
(147, 17, 111, 'share', '2024-03-19 17:18:15', NULL),
(148, 1, 105, 'view', '2024-03-19 19:09:40', 24),
(149, 3, 109, 'view', '2024-03-19 21:00:21', 55),
(150, 8, 114, 'view', '2024-03-19 22:12:19', 28),
(151, 8, 114, 'like', '2024-03-19 22:13:02', NULL),
(152, 9, 102, 'view', '2024-03-20 07:52:14', 17),
(153, 7, 104, 'view', '2024-03-20 16:33:37', 51),
(154, 7, 104, 'like', '2024-03-20 16:34:48', NULL),
(155, 7, 104, 'comment', '2024-03-20 16:35:06', NULL),
(156, 18, 110, 'view', '2024-03-20 19:03:05', 22),
(157, 18, 110, 'view', '2024-03-20 19:04:12', 2),
(158, 18, 110, 'share', '2024-03-20 19:04:24', NULL),
(159, 14, 103, 'view', '2024-03-20 20:52:39', 67),
(160, 14, 103, 'like', '2024-03-20 20:53:56', NULL),
(161, 12, 104, 'view', '2024-03-21 22:56:52', 59),
(162, 12, 104, 'comment', '2024-03-21 22:58:01', NULL),
(163, 12, 103, 'view', '2024-03-22 09:23:32', 59),
(164, 13, 114, 'view', '2024-03-22 21:15:28', 21),
(165, 13, 107, 'view', '2024-03-22 21:16:24', 8),
(166, 13, 107, 'like', '2024-03-22 21:16:44', NULL),
(167, 13, 107, 'share', '2024-03-22 21:16:59', NULL),
(168, 6, 110, 'view', '2024-03-23 08:31:17', 25),
(169, 6, 110, 'like', '2024-03-23 08:31:59', NULL),
(170, 6, 104, 'view', '2024-03-23 08:32:58', 13),
(171, 6, 104, 'like', '2024-03-23 08:33:19', NULL),
(172, 12, 106, 'view', '2024-03-23 13:02:23', 11),
(173, 12, 106, 'like', '2024-03-23 13:02:43', NULL),
(174, 12, 106, 'comment', '2024-03-23 13:02:50', NULL),
(175, 12, 106, 'view', '2024-03-23 13:03:47', 28),
(176, 3, 111, 'view', '2024-03-23 13:59:19', 21),
(177, 1, 108, 'view', '2024-03-23 17:47:16', 15),
(178, 14, 111, 'view', '2024-03-23 21:05:47', 30),
(179, 14, 111, 'like', '2024-03-23 21:06:21', NULL),
(180, 14, 106, 'view', '2024-03-23 21:06:33', 59),
(181, 3, 101, 'view', '2024-03-24 07:16:02', 4),
(182, 3, 104, 'view', '2024-03-24 07:16:41', 59),
(183, 3, 104, 'like', '2024-03-24 07:17:54', NULL),
(184, 13, 106, 'view', '2024-03-24 09:59:46', 51),
(185, 13, 106, 'comment', '2024-03-24 10:00:54', NULL),
(186, 20, 107, 'view', '2024-03-24 10:49:51', 38),
(187, 20, 107, 'share', '2024-03-24 10:50:42', NULL),
(188, 14, 104, 'view', '2024-03-24 15:18:41', 12),
(189, 14, 104, 'comment', '2024-03-24 15:19:02', NULL),
(190, 14, 112, 'view', '2024-03-24 15:20:00', 8),
(191, 1, 102, 'view', '2024-03-24 16:41:55', 24),
(192, 5, 108, 'view', '2024-03-24 18:06:24', 10),
(193, 5, 108, 'like', '2024-03-24 18:06:35', NULL),
(194, 18, 104, 'view', '2024-03-24 18:50:09', 53),
(195, 18, 114, 'view', '2024-03-24 18:51:45', 4),
(196, 12, 102, 'view', '2024-03-24 22:04:26', 27),
(197, 12, 111, 'view', '2024-03-24 22:05:03', 12),
(198, 9, 106, 'view', '2024-03-25 11:38:15', 57),
(199, 9, 106, 'like', '2024-03-25 11:39:24', NULL),
(200, 9, 104, 'view', '2024-03-25 11:40:01', 52),
(201, 9, 104, 'share', '2024-03-25 11:40:55', NULL),
(202, 14, 111, 'view', '2024-03-25 19:58:54', 44),
(203, 13, 108, 'view', '2024-03-26 10:12:08', 6),
(204, 3, 112, 'view', '2024-03-26 14:21:12', 25),
(205, 8, 101, 'view', '2024-03-26 19:03:13', 78),
(206, 8, 101, 'like', '2024-03-26 19:04:33', NULL),
(207, 5, 108, 'view', '2024-03-27 09:47:47', 6),
(208, 7, 104, 'view', '2024-03-27 16:03:01', 65),
(209, 7, 104, 'share', '2024-03-27 16:04:14', NULL),
(210, 7, 107, 'view', '2024-03-27 16:05:01', 25),
(211, 7, 107, 'like', '2024-03-27 16:05:28', NULL),
(212, 6, 114, 'view', '2024-03-27 20:45:48', 17),
(213, 6, 114, 'share', '2024-03-27 20:46:22', NULL),
(214, 18, 102, 'view', '2024-03-27 21:42:56', 19),
(215, 18, 102, 'like', '2024-03-27 21:43:29', NULL),
(216, 17, 104, 'view', '2024-03-28 12:45:50', 68),
(217, 17, 104, 'like', '2024-03-28 12:47:17', NULL),
(218, 1, 107, 'view', '2024-03-28 13:51:15', 49),
(219, 13, 110, 'view', '2024-03-28 15:10:20', 19),
(220, 13, 104, 'view', '2024-03-28 15:11:14', 77),
(221, 13, 104, 'like', '2024-03-28 15:12:48', NULL),
(222, 3, 102, 'view', '2024-03-28 20:10:03', 23),
(223, 3, 110, 'view', '2024-03-28 20:10:49', 17),
(224, 3, 110, 'comment', '2024-03-28 20:11:12', NULL),
(225, 1, 104, 'view', '2024-03-29 15:30:16', 90),
(226, 13, 107, 'view', '2024-03-29 15:43:20', 12),
(227, 3, 104, 'view', '2024-03-29 17:35:20', 6),
(228, 3, 106, 'view', '2024-03-29 17:35:31', 26),
(229, 3, 106, 'like', '2024-03-29 17:36:06', NULL),
(230, 20, 111, 'view', '2024-03-30 13:57:09', 2),
(231, 20, 111, 'comment', '2024-03-30 13:57:15', NULL),
(232, 20, 102, 'view', '2024-03-30 13:58:14', 8),
(233, 20, 102, 'share', '2024-03-30 13:58:41', NULL),
(234, 15, 109, 'view', '2024-03-30 22:50:44', 102),
(235, 15, 109, 'like', '2024-03-30 22:52:37', NULL),
(236, 6, 112, 'view', '2024-03-31 10:52:46', 22),
(237, 6, 112, 'share', '2024-03-31 10:53:10', NULL),
(238, 3, 101, 'view', '2024-03-31 11:25:37', 52),
(239, 3, 101, 'like', '2024-03-31 11:26:39', NULL),
(240, 3, 101, 'share', '2024-03-31 11:26:56', NULL),
(241, 14, 102, 'view', '2024-03-31 19:39:37', 20),
(242, 14, 102, 'share', '2024-03-31 19:40:01', NULL);
""")

print("tables:", ", ".join(r[0] for r in db.execute(
    "SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name")))

---
### Q1. Newer employees

*easy, about 3 min*

List employees hired on or after 1 January 2003. Return `FirstName`, `LastName`, `Title` and `hired` (the date only, `YYYY-MM-DD`), earliest hire first (ties by last name).

Expected: 5 rows.

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: a simple filter and sort. Pattern: basics (WHERE on an ISO date string, ORDER BY).

</details>

<details><summary><b>Hint 2</b></summary>

1. WHERE HireDate >= '2003-01-01'. 2. DATE(HireDate) drops the time part. 3. ORDER BY HireDate, LastName.

</details>

<details><summary><b>Solution</b></summary>

```sql
SELECT FirstName, LastName, Title, DATE(HireDate) AS hired
FROM Employee
WHERE HireDate >= '2003-01-01'
ORDER BY HireDate, LastName
```

Result:

| FirstName | LastName | Title | hired |
|---|---|---|---|
| Margaret | Park | Sales Support Agent | 2003-05-03 |
| Steve | Johnson | Sales Support Agent | 2003-10-17 |
| Michael | Mitchell | IT Manager | 2003-10-17 |
| Robert | King | IT Staff | 2004-01-02 |
| Laura | Callahan | IT Staff | 2004-03-04 |

**Why:** SQLite stores these dates as ISO text (`2003-05-03 00:00:00`), and ISO strings sort like dates, so a string comparison is correct. In PostgreSQL the column would be a timestamp and the same literal works.

**Complexity:** One scan of Employee.

**Common mistakes:** Comparing with '1/1/2003' (not ISO, string order breaks). `>` instead of `>=` (misses a hire on the day).

**Learn more:** [sql-basics](https://mahsa-agz.github.io/ds-handbook/#sql-basics)

</details>

---
### Q2. Most prolific composers

*easy, about 4 min*

Return the 5 composers with the most tracks: `Composer` and `tracks`, most first (ties by name). Tracks with no composer (NULL) must not count as a composer.

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: count per value, keep the top few. Pattern: GROUP BY, ORDER BY the count, LIMIT; filter NULL first.

</details>

<details><summary><b>Hint 2</b></summary>

1. WHERE Composer IS NOT NULL. 2. GROUP BY Composer. 3. ORDER BY COUNT(*) DESC, Composer. 4. LIMIT 5.

</details>

<details><summary><b>Solution</b></summary>

```sql
SELECT Composer, COUNT(*) AS tracks
FROM Track
WHERE Composer IS NOT NULL
GROUP BY Composer
ORDER BY tracks DESC, Composer
LIMIT 5
```

Result:

| Composer | tracks |
|---|---|
| Steve Harris | 80 |
| U2 | 44 |
| Jagger/Richards | 35 |
| Billy Corgan | 31 |
| Kurt Cobain | 26 |

**Why:** GROUP BY puts all NULLs into one group, which would be the 'top composer' here. Filtering them in WHERE removes that fake group.

**Complexity:** One scan of Track plus a hash per composer.

**Common mistakes:** Forgetting the NULL filter (a NULL row on top). Note the data quirk: `Composer` is free text, so 'Jagger/Richards' and 'Keith Richards/Mick Jagger' are different groups.

**Learn more:** [sql-aggregation](https://mahsa-agz.github.io/ds-handbook/#sql-aggregation)

</details>

---
### Q3. Revenue per support rep

*easy, about 5 min*

Credit each invoice to the support rep of its customer. Return `rep` (full name), `invoices` and `revenue` (2 decimals), highest revenue first.

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: a chain Employee -> Customer -> Invoice. Pattern: INNER JOINs, then GROUP BY the employee.

</details>

<details><summary><b>Hint 2</b></summary>

1. Employee e JOIN Customer c ON c.SupportRepId = e.EmployeeId. 2. JOIN Invoice i ON i.CustomerId = c.CustomerId. 3. GROUP BY e.EmployeeId. 4. COUNT and SUM.

</details>

<details><summary><b>Solution</b></summary>

```sql
SELECT e.FirstName || ' ' || e.LastName AS rep,
       COUNT(i.InvoiceId) AS invoices,
       ROUND(SUM(i.Total), 2) AS revenue
FROM Employee e
JOIN Customer c ON c.SupportRepId = e.EmployeeId
JOIN Invoice i  ON i.CustomerId = c.CustomerId
GROUP BY e.EmployeeId, rep
ORDER BY revenue DESC
```

Result:

| rep | invoices | revenue |
|---|---|---|
| Jane Peacock | 146 | 833.04 |
| Margaret Park | 140 | 775.4 |
| Steve Johnson | 126 | 720.16 |

**Why:** Each invoice has one customer and each customer one rep, so the joins do not duplicate invoices.

**Complexity:** O(Employee + Customer + Invoice).

**Common mistakes:** Joining Customer to Employee on EmployeeId = CustomerId. Grouping by name only.

**Learn more:** [sql-joins](https://mahsa-agz.github.io/ds-handbook/#sql-joins)

</details>

---
### Q4. Categories with dead videos

*easy, about 5 min*

On the made-up product tables: for each `category`, return `videos` (number of videos) and `videos_without_views` (videos with no view event at all). Most dead videos first, then by category. Every category must appear, even with 0.

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: 'with no view' plus every category kept. Pattern: LEFT JOIN with the filter in ON, then counts that ignore NULL.

</details>

<details><summary><b>Hint 2</b></summary>

1. videos LEFT JOIN events ON video_id AND event_type = 'view'. 2. GROUP BY category. 3. COUNT(DISTINCT v.video_id) all videos; COUNT(DISTINCT e.video_id) videos that matched. 4. Subtract.

</details>

<details><summary><b>Solution</b></summary>

```sql
SELECT v.category,
       COUNT(DISTINCT v.video_id) AS videos,
       COUNT(DISTINCT v.video_id) - COUNT(DISTINCT e.video_id) AS videos_without_views
FROM videos v
LEFT JOIN events e
       ON e.video_id = v.video_id
      AND e.event_type = 'view'
GROUP BY v.category
ORDER BY videos_without_views DESC, v.category
```

Result:

| category | videos | videos_without_views |
|---|---|---|
| gaming | 3 | 2 |
| comedy | 3 | 0 |
| education | 2 | 0 |
| food | 5 | 0 |
| pets | 2 | 0 |

**Why:** The join creates many rows per watched video, so plain COUNT(*) is useless; COUNT(DISTINCT) brings it back to videos. COUNT(DISTINCT e.video_id) ignores the NULL from unmatched videos.

**Complexity:** O(videos + events).

**Common mistakes:** Putting `event_type = 'view'` in WHERE (dead videos disappear). COUNT(*) instead of COUNT(DISTINCT).

**Learn more:** [sql-left-join-nulls](https://mahsa-agz.github.io/ds-handbook/#sql-left-join-nulls)

</details>

---
### Q5. Active users by country

*easy, about 5 min*

On the made-up product tables: for each country (show `unknown` for NULL), return `users` (all users) and `active_users` (users with at least one event). Most users first, then by country.

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: a flag per user (active or not) counted per group. Pattern: LEFT JOIN to the distinct active users, then SUM(CASE WHEN ...).

</details>

<details><summary><b>Hint 2</b></summary>

1. Derived table: SELECT DISTINCT user_id FROM events. 2. users LEFT JOIN it. 3. GROUP BY COALESCE(country, 'unknown'). 4. COUNT(*) and SUM(CASE WHEN a.user_id IS NOT NULL THEN 1 ELSE 0 END).

</details>

<details><summary><b>Solution</b></summary>

```sql
SELECT COALESCE(u.country, 'unknown') AS country,
       COUNT(*) AS users,
       SUM(CASE WHEN a.user_id IS NOT NULL THEN 1 ELSE 0 END) AS active_users
FROM users u
LEFT JOIN (SELECT DISTINCT user_id FROM events) a ON a.user_id = u.user_id
GROUP BY COALESCE(u.country, 'unknown')
ORDER BY users DESC, country
```

Result:

| country | users | active_users |
|---|---|---|
| KR | 5 | 4 |
| US | 4 | 4 |
| BR | 3 | 2 |
| DE | 3 | 2 |
| ID | 3 | 3 |
| unknown | 2 | 2 |

**Why:** Deduplicating events first keeps one row per user, so COUNT(*) still counts users. The CASE turns 'matched or not' into 1 or 0.

**Complexity:** O(users + events).

**Common mistakes:** Joining raw events (each user repeated per event, so `users` is inflated). Grouping by `country` and showing COALESCE: that works, but group by the same expression you show to avoid surprises.

**Learn more:** [sql-case-when](https://mahsa-agz.github.io/ds-handbook/#sql-case-when), [sql-left-join-nulls](https://mahsa-agz.github.io/ds-handbook/#sql-left-join-nulls)

</details>

---
### Q6. Customers who never bought Metal

*easy, about 8 min*

Metal is the third best-selling genre. Which customers have **never** bought a Metal track? Return `CustomerId`, `FirstName`, `LastName`, `Country`, by CustomerId. Expected: 4 rows.

Follow-up to answer out loud: why can `NOT IN` be dangerous here?

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: 'never' about related rows several joins away. Pattern: NOT EXISTS with a correlated subquery.

</details>

<details><summary><b>Hint 2</b></summary>

1. Outer: FROM Customer c. 2. Inner: Invoice -> InvoiceLine -> Track -> Genre, WHERE i.CustomerId = c.CustomerId AND g.Name = 'Metal'. 3. WHERE NOT EXISTS (inner).

</details>

<details><summary><b>Solution</b></summary>

```sql
SELECT c.CustomerId, c.FirstName, c.LastName, c.Country
FROM Customer c
WHERE NOT EXISTS (
    SELECT 1
    FROM Invoice i
    JOIN InvoiceLine il ON il.InvoiceId = i.InvoiceId
    JOIN Track t ON t.TrackId = il.TrackId
    JOIN Genre g ON g.GenreId = t.GenreId
    WHERE i.CustomerId = c.CustomerId
      AND g.Name = 'Metal')
ORDER BY c.CustomerId
```

Result:

| CustomerId | FirstName | LastName | Country |
|---|---|---|---|
| 4 | Bjørn | Hansen | Norway |
| 6 | Helena | Holý | Czech Republic |
| 43 | Isabelle | Mercier | France |
| 49 | Stanisław | Wójcik | Poland |

**Why:** NOT EXISTS is true when the inner query finds no row for this customer. Follow-up: `CustomerId NOT IN (subquery)` returns no rows at all if the subquery yields a NULL, because `x NOT IN (1, NULL)` is unknown. Here the ids are never NULL, but NOT EXISTS is safe by design.

**Complexity:** Hash anti-join: O(Customer + InvoiceLine).

**Common mistakes:** `WHERE g.Name <> 'Metal'` after a join: that finds customers who bought something that is not Metal, which is a different (much bigger) set.

**Learn more:** [sql-subqueries](https://mahsa-agz.github.io/ds-handbook/#sql-subqueries)

</details>

---
## Redo list

Write down every question you got wrong, solved only after Hint 2, or finished over time. Solve it again from scratch 3 days and 7 days later.

**Next:** [Statistics, A/B testing and product](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_07/3_stats.ipynb)